In [20]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Load dataset
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/nlp_dataset.csv')
print(df.head())

# Text Cleaning Function
def clean_text(text):
    text = text.lower()                           # lowercase
    text = re.sub(r'[^a-z\s]', '', text)          # remove punctuation/numbers
    tokens = nltk.word_tokenize(text)                  # tokenize
    tokens = [w for w in tokens if w not in stopwords.words('english')]
    return " ".join(tokens)

df['clean_text'] = df['Comment'].apply(clean_text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                                             Comment Emotion
0  i seriously hate one subject to death but now ...    fear
1                 im so full of life i feel appalled   anger
2  i sit here to write i start to dig out my feel...    fear
3  ive been really angry with r and i feel like a...     joy
4  i feel suspicious if there is no one outside l...    fear


Explanation:

Lowercasing ensures uniformity.

Removing punctuation/noise improves signal.

Tokenization + stopword removal reduces irrelevant words.
Impact: Cleaner input → better feature extraction → improved model accuracy.

**Feature Extraction**

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text']).toarray()
y = df['Emotion']   # target column

print("Feature shape:", X.shape)

Feature shape: (5937, 5000)


Explanation:

TF-IDF assigns higher weight to rare but important words.

Converts text → numerical matrix usable by ML models.

Example: “happy” gets higher weight in emotion classification than common words like “the”

**Model Development**

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)

# Support Vector Machine
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)


**Mondel Compariso**

In [26]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_preds))
print("Naive Bayes F1:", f1_score(y_test, nb_preds, average='weighted'))

print("SVM Accuracy:", accuracy_score(y_test, svm_preds))
print("SVM F1:", f1_score(y_test, svm_preds, average='weighted'))

print("\nClassification Report (SVM):\n", classification_report(y_test, svm_preds))


Naive Bayes Accuracy: 0.9090909090909091
Naive Bayes F1: 0.9089193266783082
SVM Accuracy: 0.946969696969697
SVM F1: 0.9469121249090242

Classification Report (SVM):
               precision    recall  f1-score   support

       anger       0.93      0.96      0.94       392
        fear       0.97      0.92      0.94       416
         joy       0.95      0.96      0.95       380

    accuracy                           0.95      1188
   macro avg       0.95      0.95      0.95      1188
weighted avg       0.95      0.95      0.95      1188



Explanation:

Naive Bayes: Fast, works well with text data, assumes independence between words.

SVM: Finds optimal hyperplane, often better for high‑dimensional sparse data like TF‑IDF.

Suitability: SVM usually outperforms NB in emotion classification due to better handling of overlapping classes.

In [27]:
import joblib

# Save the best performing model (assume SVM here)
joblib.dump(svm_model, "best_model.pkl")


['best_model.pkl']